# Realistic LLM Use Case: When LLMs Actually Win
**SNIA DSN AI Stack Webinar Series — 2026**

This notebook demonstrates a realistic storage engineering task — classifying **unstructured incident report messages** into 6 categories — where LLM fine-tuning meaningfully outperforms traditional ML.

### Key Finding
TF-IDF + XGBoost achieves ~70–80%. SFT achieves ~85–92%. The gap comes from shared vocabulary across categories that only contextual understanding can disambiguate.

### Dataset
720 synthetic incident reports (120 per category, 6 categories) with deliberately overlapping vocabulary.

## Two Modes

| Mode | What happens | Time |
|------|-------------|------|
| **Full Training** (Step 8A) | Train LoRA adapter from scratch on GPU | ~20 min on T4 |
| **Demo** (Step 8B) | Load pre-trained adapter from HuggingFace Hub | ~2 min |

## Prerequisites
- GPU runtime (T4 or A100) — Go to **Runtime > Change runtime type > T4 GPU**
- HuggingFace token (optional, for Step 8A model upload)

## How to Use
1. Run Steps 1–7 (data generation and baseline)
2. Choose **either** Step 8A (full training) **or** Step 8B (demo mode) — not both
3. Run Steps 9–12 (evaluation and analysis)

## Step 1: Prerequisites and HuggingFace Setup

Install all required packages, check GPU availability, and configure the HuggingFace token.

A HuggingFace token is needed for **Step 8A** to upload the trained model after training completes. If you only plan to use **Step 8B** (demo mode), you can leave the token blank.

To get your token:
1. Go to [huggingface.co/settings/tokens](https://huggingface.co/settings/tokens)
2. Create a new token with **write** access
3. Paste it in the `HF_TOKEN` variable below

In [ ]:
# Step 1: Prerequisites and HuggingFace Setup
!pip install -q torch transformers datasets accelerate peft trl scikit-learn xgboost matplotlib seaborn sentencepiece huggingface_hub

import torch
import time as _time

_notebook_start = _time.time()

print(f"GPU available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    mem = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f"Memory: {mem:.1f} GB")
else:
    print("WARNING: GPU recommended for SFT training.")
    print("Go to Runtime > Change runtime type > T4 GPU")

# HuggingFace token (required for Step 8A model upload)
# Get your token from: https://huggingface.co/settings/tokens
HF_TOKEN = ""  # <-- Paste your token here

# Repository for the pre-trained adapter
HF_REPO = "gidft/smollm2-error-classifier-sft"

## Step 2: Define Error Categories

Six categories of storage infrastructure incident reports, each with 8 multi-line template variations at three difficulty tiers. Templates are realistic multi-line syslog / storage-manager output where classification emerges from the RELATIONSHIP between log lines, not from any single keyword.

| Category | What it covers |
|----------|---------------|
| Drive Failure | SMART failures, media errors, device timeouts, RAID degradation |
| Network Issue | Link errors, path instability, fabric problems, connectivity failures |
| Capacity Warning | Pool usage, space exhaustion, thin provisioning, growth projections |
| Performance Degradation | Latency spikes, throughput drops, workload contention |
| Configuration Error | Setting changes, parameter reversions, routing errors |
| Firmware Bug | Vendor defects, version-specific bugs, false positive health alerts |

**Difficulty tiers per category:**
- ~3 **EASY**: category-specific language that TF-IDF can latch onto
- ~3 **MEDIUM**: shared vocabulary but context disambiguates
- ~2 **HARD**: cross-category causation or negation that confuses keyword-based classifiers

In [ ]:
# Step 2: Define Error Categories
import random
import numpy as np
from collections import Counter
from datetime import datetime, timedelta

random.seed(42)
np.random.seed(42)

# ---------------------------------------------------------------------------
# Shared variable pools (identical across all categories to prevent leakage)
# ---------------------------------------------------------------------------
SHARED_VARS = {
    "dev":          ["sda", "sdb", "sdc", "sdd", "sde", "nvme0n1", "nvme1n1", "nvme2n1"],
    "node":         ["stor-node01", "stor-node02", "stor-node03", "stor-node04"],
    "node2":        ["stor-node05", "stor-node06", "stor-node07", "stor-node08"],
    "port":         ["eth0", "eth1", "bond0", "ib0", "fc0", "fc1"],
    "ctrl":         ["A", "B"],
    "pool":         ["pool0", "pool1", "tank0", "data-pool"],
    "vol":          ["vol-prod-01", "vol-db-02", "vol-archive-03", "vol-vdi-04"],
    "count":        [1048576, 2097152, 4194304, 524288, 8388608, 16777216],
    "hours":        [12, 24, 48, 72, 168, 720],
    "mins":         [3, 5, 8, 12, 15, 22, 30, 45],
    "pct":          [78, 82, 85, 88, 91, 93, 95, 97],
    "ms":           [45, 68, 92, 120, 180, 250, 340, 500],
    "baseline_ms":  [8, 12, 15, 18, 22, 25],
    "ver":          ["3.2.1", "3.2.4", "3.3.0", "4.0.1", "4.1.0", "4.1.2"],
    "incident":     ["INC-4821", "INC-5033", "INC-5190", "INC-5402", "INC-5617"],
    "threshold":    [80, 85, 90, 95],
    "ratio":        [0.12, 0.18, 0.25, 0.34, 0.45],
    "rate":         [1.2, 2.5, 3.8, 5.1, 7.4, 12.6],
    "size_tb":      [4.8, 9.6, 14.4, 19.2, 28.8, 48.0],
    "rg":           ["rg0", "rg1", "rg2", "rg3"],
    "host":         ["esxi-host01", "esxi-host02", "k8s-worker03", "db-server04"],
    "site":         ["site-east", "site-west", "site-central"],
}


def generate_timestamps(n=5, base=None, interval_sec=(1, 30)):
    """Return n ISO-ish syslog timestamps separated by realistic intervals."""
    if base is None:
        base = datetime(2026, 3, 28, 14, 22, 11)
    stamps = []
    t = base
    for _ in range(n):
        stamps.append(t.strftime("%Y-%m-%dT%H:%M:%S"))
        t += timedelta(seconds=random.randint(*interval_sec))
    return stamps


def pick(pool_name):
    """Pick a random value from a shared variable pool."""
    return str(random.choice(SHARED_VARS[pool_name]))


# ---------------------------------------------------------------------------
# TEMPLATES: 6 categories x 8 templates = 48 total
# ---------------------------------------------------------------------------

TEMPLATES = {
    # ==================================================================
    # DRIVE FAILURE
    # ==================================================================
    "Drive Failure": {
        "templates": [
            # --- EASY 1: clear device I/O errors + SMART failure ---
            "{ts1} {node} kernel: {dev}: I/O error, sector {count}, cmd 0x2a\n"
            "{ts2} {node} smartd: Device /dev/{dev}: SMART Health Status: FAILED\n"
            "{ts3} {node} smartd: Device /dev/{dev}: reallocated sector count = 148, threshold = 5\n"
            "{ts4} {node} kernel: {dev}: media error on read, LBA {count}",

            # --- EASY 2: device timeout + SMART wear-out ---
            "{ts1} {node} kernel: {dev}: cmd 0x28 timeout after 30s\n"
            "{ts2} {node} smartd: Device /dev/{dev}: wear leveling count = 2, min threshold = 10\n"
            "{ts3} {node} kernel: {dev}: device not responding, resetting link\n"
            "{ts4} {node} ctrl-mgr: controller {ctrl} initiated device reseat for /dev/{dev}",

            # --- EASY 3: repeated I/O errors + uncorrectable count ---
            "{ts1} {node} kernel: {dev}: I/O error, sector {count}, cmd 0x2a\n"
            "{ts2} {node} kernel: {dev}: I/O error, sector {count}, cmd 0x28\n"
            "{ts3} {node} smartd: Device /dev/{dev}: uncorrectable error count = 37\n"
            "{ts4} {node} storage-mgr: {vol} marked degraded, rebuild to spare started",

            # --- MEDIUM 1: device errors + latency spike (looks like perf degradation) ---
            "{ts1} {node} storage-mgr: {vol} latency p99={ms}ms (baseline: {baseline_ms}ms)\n"
            "{ts2} {node} kernel: {dev}: I/O error, sector {count}, cmd 0x2a\n"
            "{ts3} {node} smartd: Device /dev/{dev}: current pending sector count = 24\n"
            "{ts4} {node} storage-mgr: {vol} throughput dropped {pct}% correlates with /dev/{dev} errors",

            # --- MEDIUM 2: device errors + network retransmissions (looks like network) ---
            "{ts1} {node} kernel: {dev}: cmd 0x2a timeout after 30s\n"
            "{ts2} {node} network-mgr: {port} retransmissions={rate}/s (elevated)\n"
            "{ts3} {node} smartd: Device /dev/{dev}: SMART Health Status: FAILED\n"
            "{ts4} {node} kernel: {dev}: media error on read, LBA {count}",

            # --- MEDIUM 3: device errors + replication lag (looks like repl issue) ---
            "{ts1} {node} repl-mgr: {vol} replication lag={mins}m to {site}\n"
            "{ts2} {node} kernel: {dev}: I/O error, sector {count}, cmd 0x28, rejected by device\n"
            "{ts3} {node} smartd: Device /dev/{dev}: reallocated sector count = 92, threshold = 5\n"
            "{ts4} {node} storage-mgr: {rg} rebuild started, source: /dev/{dev}",

            # --- HARD 1: device causing controller symptoms (looks like firmware/config) ---
            "{ts1} {node} ctrl-mgr: controller {ctrl} queue depth reached 512, fw={ver}\n"
            "{ts2} {node} ctrl-mgr: controller {ctrl} memory utilization {pct}%\n"
            "{ts3} {node} kernel: {dev}: I/O error, sector {count}, cmd 0x2a\n"
            "{ts4} {node} smartd: Device /dev/{dev}: uncorrectable error count = 64\n"
            "{ts5} {node} ctrl-mgr: controller {ctrl} queue depth returned to normal after /dev/{dev} removed",

            # --- HARD 2: device errors misattributed to firmware (version mentioned but irrelevant) ---
            "{ts1} {node} ctrl-mgr: controller {ctrl} fw={ver} loaded {hours}h ago\n"
            "{ts2} {node} kernel: {dev}: cmd 0x28 timeout after 30s\n"
            "{ts3} {node} smartd: Device /dev/{dev}: wear leveling count = 1, min threshold = 10\n"
            "{ts4} {node} kernel: {dev}: vendor confirmed drive model end-of-life, spare pool exhausted\n"
            "{ts5} {node} storage-mgr: same firmware version running stable on {node2} without /dev/{dev}",
        ],
    },

    # ==================================================================
    # NETWORK ISSUE
    # ==================================================================
    "Network Issue": {
        "templates": [
            # --- EASY 1: clear link errors + fabric event ---
            "{ts1} {node} network-mgr: {port} link flap detected, down/up in {mins}s\n"
            "{ts2} {node} network-mgr: fabric event: switch port renegotiated at reduced speed\n"
            "{ts3} {node} network-mgr: {port} CRC errors={count} in last {hours}h",

            # --- EASY 2: path failure + retransmissions ---
            "{ts1} {node} network-mgr: {port} path to {node2} marked DOWN\n"
            "{ts2} {node} network-mgr: {port} retransmissions={rate}/s, baseline=0.01/s\n"
            "{ts3} {node} network-mgr: failover to alternate path completed, traffic reverted to {port}",

            # --- EASY 3: multiple link errors + packet loss ---
            "{ts1} {node} network-mgr: {port} RX errors={count} in last {hours}h\n"
            "{ts2} {node} network-mgr: {port} packet loss={ratio}% over {mins}m window\n"
            "{ts3} {node} network-mgr: {port} link speed changed from 100Gbps to 25Gbps\n"
            "{ts4} {node} network-mgr: fabric event: upstream switch reported port errors",

            # --- MEDIUM 1: network causing device timeouts (looks like drive failure) ---
            "{ts1} {node} kernel: {dev}: cmd 0x28 timeout after 30s\n"
            "{ts2} {node} network-mgr: {port} retransmissions={rate}/s, baseline=0.01/s\n"
            "{ts3} {node} network-mgr: {port} path to storage target marked DOWN\n"
            "{ts4} {node} smartd: Device /dev/{dev}: SMART Health Status: OK",

            # --- MEDIUM 2: network causing latency spike (looks like perf degradation) ---
            "{ts1} {node} storage-mgr: {vol} latency p99={ms}ms (baseline: {baseline_ms}ms)\n"
            "{ts2} {node} network-mgr: {port} retransmissions={rate}/s, baseline=0.01/s\n"
            "{ts3} {node} storage-mgr: {vol} throughput dropped {pct}% in last {mins}m\n"
            "{ts4} {node} network-mgr: latency spike correlates with {port} CRC errors={count} in last {hours}h",

            # --- MEDIUM 3: network causing replication failures (looks like repl issue) ---
            "{ts1} {node} repl-mgr: {vol} sync to {site} stalled for {mins}m\n"
            "{ts2} {node} network-mgr: {port} path to {site} marked DOWN, send buffer exhausted\n"
            "{ts3} {node} network-mgr: {port} packet loss={ratio}% over {mins}m window\n"
            "{ts4} {node} repl-mgr: {vol} sync resumed after alternate path activated",

            # --- HARD 1: network issue mentioning firmware + capacity (red herrings) ---
            "{ts1} {node} ctrl-mgr: controller {ctrl} fw={ver} loaded {hours}h ago\n"
            "{ts2} {node} storage-mgr: {pool} utilization={pct}% of {size_tb}TB\n"
            "{ts3} {node} network-mgr: {port} link flap detected, down/up in {mins}s\n"
            "{ts4} {node} network-mgr: {port} CRC errors={count} in last {hours}h\n"
            "{ts5} {node} storage-mgr: {vol} latency normalized after {port} path restored",

            # --- HARD 2: network with controller memory spike (looks like firmware bug) ---
            "{ts1} {node} ctrl-mgr: controller {ctrl} memory utilization {pct}%\n"
            "{ts2} {node} network-mgr: {port} retransmissions={rate}/s causing TCP window collapse\n"
            "{ts3} {node} ctrl-mgr: controller {ctrl} retry queue backed up, depth=4096\n"
            "{ts4} {node} network-mgr: vendor confirmed optic degradation on {port}, path to {node2} marked DOWN\n"
            "{ts5} {node} ctrl-mgr: controller {ctrl} memory utilization returned to baseline after {port} failover",
        ],
    },

    # ==================================================================
    # CAPACITY WARNING
    # ==================================================================
    "Capacity Warning": {
        "templates": [
            # --- EASY 1: pool utilization + quota alert ---
            "{ts1} {node} storage-mgr: {pool} utilization={pct}% of {size_tb}TB\n"
            "{ts2} {node} storage-mgr: {pool} exceeded threshold at {threshold}%\n"
            "{ts3} {node} storage-mgr: {vol} quota alert: {size_tb}TB of {size_tb}TB used",

            # --- EASY 2: thin provision + snapshot reserve exhaustion ---
            "{ts1} {node} storage-mgr: {pool} thin provision reserve depleted\n"
            "{ts2} {node} storage-mgr: {pool} snapshot reserve usage={pct}%\n"
            "{ts3} {node} storage-mgr: {vol} auto-grow failed, maximum volume size reached\n"
            "{ts4} {node} storage-mgr: {pool} utilization={pct}% of {size_tb}TB",

            # --- EASY 3: pool full + write rejection ---
            "{ts1} {node} storage-mgr: {pool} utilization={pct}% of {size_tb}TB\n"
            "{ts2} {node} storage-mgr: {pool} write operations throttled, free space below minimum\n"
            "{ts3} {node} storage-mgr: {vol} write rejected: pool space exhausted",

            # --- MEDIUM 1: capacity causing write I/O errors (looks like drive failure) ---
            "{ts1} {node} kernel: {dev}: I/O error, sector {count}, cmd 0x2a\n"
            "{ts2} {node} storage-mgr: {pool} utilization={pct}% of {size_tb}TB\n"
            "{ts3} {node} smartd: Device /dev/{dev}: SMART Health Status: OK\n"
            "{ts4} {node} storage-mgr: {pool} write operations throttled, free space below minimum",

            # --- MEDIUM 2: capacity causing latency spikes (looks like perf degradation) ---
            "{ts1} {node} storage-mgr: {vol} latency p99={ms}ms (baseline: {baseline_ms}ms)\n"
            "{ts2} {node} storage-mgr: {pool} utilization={pct}% of {size_tb}TB\n"
            "{ts3} {node} storage-mgr: {host} presenting {count} concurrent I/O requests during rebuild\n"
            "{ts4} {node} storage-mgr: {pool} garbage collection backlog={count} segments",

            # --- MEDIUM 3: capacity causing replication stall (looks like repl/network) ---
            "{ts1} {node} repl-mgr: {vol} replication lag={mins}m to {site}\n"
            "{ts2} {node} storage-mgr: {pool} utilization={pct}% of {size_tb}TB\n"
            "{ts3} {node} storage-mgr: {pool} snapshot reserve usage={pct}%\n"
            "{ts4} {node} repl-mgr: {vol} sync stalled, destination pool space exhausted",

            # --- HARD 1: capacity issue mentioning firmware version (red herring) ---
            "{ts1} {node} ctrl-mgr: controller {ctrl} fw={ver} loaded {hours}h ago\n"
            "{ts2} {node} storage-mgr: {pool} utilization={pct}% of {size_tb}TB\n"
            "{ts3} {node} storage-mgr: {vol} latency p99={ms}ms (baseline: {baseline_ms}ms)\n"
            "{ts4} {node} storage-mgr: {pool} exceeded threshold at {threshold}%\n"
            "{ts5} {node} storage-mgr: same firmware version on {node2} operating normally at lower utilization",

            # --- HARD 2: capacity issue mentioning network paths (red herring) ---
            "{ts1} {node} network-mgr: {port} path to {site} status: OK\n"
            "{ts2} {node} storage-mgr: {pool} utilization={pct}% of {size_tb}TB\n"
            "{ts3} {node} kernel: {dev}: I/O error, sector {count}, cmd 0x2a\n"
            "{ts4} {node} smartd: Device /dev/{dev}: SMART Health Status: OK\n"
            "{ts5} {node} storage-mgr: {pool} write operations throttled, free space below minimum",
        ],
    },

    # ==================================================================
    # PERFORMANCE DEGRADATION
    # ==================================================================
    "Performance Degradation": {
        "templates": [
            # --- EASY 1: workload contention ---
            "{ts1} {node} storage-mgr: {vol} latency p99={ms}ms (baseline: {baseline_ms}ms)\n"
            "{ts2} {node} storage-mgr: {vol} IOPS={count}, exceeds expected workload envelope\n"
            "{ts3} {node} ctrl-mgr: controller {ctrl} queue depth={count}, traffic at saturation",

            # --- EASY 2: cache resource saturation ---
            "{ts1} {node} storage-mgr: {vol} read cache hit ratio={ratio} (baseline: 0.92)\n"
            "{ts2} {node} storage-mgr: {vol} latency p99={ms}ms (baseline: {baseline_ms}ms)\n"
            "{ts3} {node} ctrl-mgr: controller {ctrl} cache eviction rate={rate}/s, capacity pressure",

            # --- EASY 3: host-side queue saturation ---
            "{ts1} {node} storage-mgr: {vol} latency p99={ms}ms (baseline: {baseline_ms}ms)\n"
            "{ts2} {node} storage-mgr: {vol} throughput dropped {pct}% in last {mins}m\n"
            "{ts3} {node} storage-mgr: {host} presenting {count} concurrent I/O requests\n"
            "{ts4} {node} ctrl-mgr: controller {ctrl} queue depth={count}, service time elevated",

            # --- MEDIUM 1: performance with device timeouts (looks like drive failure) ---
            "{ts1} {node} storage-mgr: {vol} latency p99={ms}ms (baseline: {baseline_ms}ms)\n"
            "{ts2} {node} kernel: {dev}: cmd 0x28 timeout after 30s\n"
            "{ts3} {node} smartd: Device /dev/{dev}: SMART Health Status: OK\n"
            "{ts4} {node} storage-mgr: all devices in {rg} showing elevated service time\n"
            "{ts5} {node} ctrl-mgr: controller {ctrl} queue depth={count}, service time elevated",

            # --- MEDIUM 2: performance with cache drops (looks like firmware issue) ---
            "{ts1} {node} storage-mgr: {vol} read cache hit ratio={ratio} (baseline: 0.92)\n"
            "{ts2} {node} ctrl-mgr: controller {ctrl} fw={ver} loaded {hours}h ago\n"
            "{ts3} {node} storage-mgr: {vol} latency p99={ms}ms (baseline: {baseline_ms}ms)\n"
            "{ts4} {node} storage-mgr: working set size exceeds cache capacity, eviction rate={rate}/s",

            # --- MEDIUM 3: performance with retransmissions (looks like network) ---
            "{ts1} {node} storage-mgr: {vol} latency p99={ms}ms (baseline: {baseline_ms}ms)\n"
            "{ts2} {node} network-mgr: {port} retransmissions={rate}/s (elevated)\n"
            "{ts3} {node} network-mgr: {port} CRC errors=0 in last {hours}h\n"
            "{ts4} {node} storage-mgr: {host} presenting {count} concurrent I/O requests\n"
            "{ts5} {node} storage-mgr: {vol} latency correlates with host submission rate, not network",

            # --- HARD 1: performance mentioning firmware update but ruling it out ---
            "{ts1} {node} ctrl-mgr: controller {ctrl} fw={ver} loaded {hours}h ago\n"
            "{ts2} {node} storage-mgr: {vol} latency p99={ms}ms (baseline: {baseline_ms}ms)\n"
            "{ts3} {node} storage-mgr: {vol} throughput dropped {pct}% in last {mins}m\n"
            "{ts4} {node} storage-mgr: {node2} running same fw={ver} with normal latency\n"
            "{ts5} {node} storage-mgr: {host} restart detected: {count} new concurrent sessions in last {mins}m",

            # --- HARD 2: performance mentioning capacity threshold but ruling it out ---
            "{ts1} {node} storage-mgr: {pool} utilization={pct}% of {size_tb}TB\n"
            "{ts2} {node} storage-mgr: {vol} latency p99={ms}ms (baseline: {baseline_ms}ms)\n"
            "{ts3} {node} ctrl-mgr: controller {ctrl} queue depth={count}, service time elevated\n"
            "{ts4} {node} storage-mgr: {pool} garbage collection operating normally\n"
            "{ts5} {node} storage-mgr: {vol} latency spike correlates with {host} batch job start",
        ],
    },

    # ==================================================================
    # CONFIGURATION ERROR
    # ==================================================================
    "Configuration Error": {
        "templates": [
            # --- EASY 1: setting changed + reverting resolved ---
            "{ts1} {node} storage-mgr: {vol} max queue depth changed from 64 to 8 at {ts1}\n"
            "{ts2} {node} storage-mgr: {vol} latency p99={ms}ms (baseline: {baseline_ms}ms)\n"
            "{ts3} {node} ctrl-mgr: controller {ctrl} queue depth={count}, service time elevated\n"
            "{ts4} {node} storage-mgr: {vol} max queue depth reverted to 64, latency returned to baseline",

            # --- EASY 2: cache policy changed + revert ---
            "{ts1} {node} storage-mgr: {vol} write-back cache disabled at {ts1}\n"
            "{ts2} {node} storage-mgr: {vol} latency p99={ms}ms (baseline: {baseline_ms}ms)\n"
            "{ts3} {node} storage-mgr: {vol} throughput dropped {pct}% in last {mins}m\n"
            "{ts4} {node} storage-mgr: {vol} write-back cache re-enabled, throughput restored",

            # --- EASY 3: MTU changed + revert ---
            "{ts1} {node} network-mgr: {port} MTU changed from 9000 to 1500 at {ts1}\n"
            "{ts2} {node} network-mgr: {port} retransmissions={rate}/s, baseline=0.01/s\n"
            "{ts3} {node} storage-mgr: {vol} latency p99={ms}ms (baseline: {baseline_ms}ms)\n"
            "{ts4} {node} network-mgr: {port} MTU reverted to 9000, retransmissions normalized",

            # --- MEDIUM 1: looks like network issue, root cause is path weight change ---
            "{ts1} {node} network-mgr: {port} path to {node2} marked DOWN\n"
            "{ts2} {node} network-mgr: {port} retransmissions={rate}/s, baseline=0.01/s\n"
            "{ts3} {node} storage-mgr: {vol} latency p99={ms}ms (baseline: {baseline_ms}ms)\n"
            "{ts4} {node} network-mgr: {port} path weight changed from 100 to 0 at {ts1}\n"
            "{ts5} {node} network-mgr: {port} path weight reverted to 100, path restored",

            # --- MEDIUM 2: looks like drive failure, root cause is I/O scheduler change ---
            "{ts1} {node} kernel: {dev}: cmd 0x28 timeout after 30s\n"
            "{ts2} {node} smartd: Device /dev/{dev}: SMART Health Status: OK\n"
            "{ts3} {node} storage-mgr: {vol} I/O scheduler changed from deadline to noop at {ts1}\n"
            "{ts4} {node} kernel: {dev}: I/O error, sector {count}, cmd 0x2a\n"
            "{ts5} {node} storage-mgr: {vol} I/O scheduler reverted to deadline, errors ceased",

            # --- MEDIUM 3: looks like capacity, root cause is tier policy change ---
            "{ts1} {node} storage-mgr: {pool} write operations throttled, free space below minimum\n"
            "{ts2} {node} storage-mgr: {pool} tiering policy changed from auto to archive-only at {ts1}\n"
            "{ts3} {node} storage-mgr: {pool} utilization={pct}% of {size_tb}TB\n"
            "{ts4} {node} storage-mgr: {pool} tiering policy reverted to auto, writes resumed",

            # --- HARD 1: looks like firmware bug -- version mentioned, but parameter changed ---
            "{ts1} {node} ctrl-mgr: controller {ctrl} fw={ver} loaded {hours}h ago\n"
            "{ts2} {node} storage-mgr: {vol} latency p99={ms}ms (baseline: {baseline_ms}ms)\n"
            "{ts3} {node} ctrl-mgr: controller {ctrl} memory utilization {pct}%\n"
            "{ts4} {node} ctrl-mgr: controller {ctrl} prefetch depth changed from 64 to 4 at {ts1}\n"
            "{ts5} {node} ctrl-mgr: controller {ctrl} prefetch depth reverted to 64, memory and latency normalized",

            # --- HARD 2: looks like perf degradation -- workload spike, but root is stripe width ---
            "{ts1} {node} storage-mgr: {vol} latency p99={ms}ms (baseline: {baseline_ms}ms)\n"
            "{ts2} {node} storage-mgr: {vol} throughput dropped {pct}% in last {mins}m\n"
            "{ts3} {node} ctrl-mgr: controller {ctrl} queue depth={count}, service time elevated\n"
            "{ts4} {node} storage-mgr: {rg} stripe width changed from 128K to 16K at {ts1}\n"
            "{ts5} {node} storage-mgr: {rg} stripe width reverted to 128K, throughput restored",
        ],
    },

    # ==================================================================
    # FIRMWARE BUG
    # ==================================================================
    "Firmware Bug": {
        "templates": [
            # --- EASY 1: version correlation + vendor confirmation ---
            "{ts1} {node} ctrl-mgr: controller {ctrl} fw={ver} loaded {hours}h ago\n"
            "{ts2} {node} storage-mgr: {vol} latency p99={ms}ms (baseline: {baseline_ms}ms)\n"
            "{ts3} {node} ctrl-mgr: controller {ctrl} memory utilization {pct}%, growing\n"
            "{ts4} {node} ctrl-mgr: vendor bulletin {incident}: fw={ver} exhibits elevated memory growth under sustained load",

            # --- EASY 2: version difference between nodes ---
            "{ts1} {node} ctrl-mgr: controller {ctrl} fw={ver} loaded {hours}h ago\n"
            "{ts2} {node} storage-mgr: {vol} throughput dropped {pct}% in last {mins}m\n"
            "{ts3} {node2} ctrl-mgr: controller {ctrl} fw=3.1.9 operating normally with same workload\n"
            "{ts4} {node} ctrl-mgr: vendor bulletin {incident}: throughput regression in fw={ver}",

            # --- EASY 3: restart cycle tied to version ---
            "{ts1} {node} ctrl-mgr: controller {ctrl} fw={ver} unexpected restart\n"
            "{ts2} {node} ctrl-mgr: controller {ctrl} fw={ver} restart count=3 in {hours}h, started after traffic spike\n"
            "{ts3} {node} ctrl-mgr: controller {ctrl} core dump generated, reference {incident}\n"
            "{ts4} {node} ctrl-mgr: vendor confirmed: fw={ver} restart under specific I/O pattern",

            # --- MEDIUM 1: looks like drive failure, but version reveals firmware ---
            "{ts1} {node} kernel: {dev}: I/O error, sector {count}, cmd 0x2a\n"
            "{ts2} {node} smartd: Device /dev/{dev}: SMART Health Status: OK\n"
            "{ts3} {node} ctrl-mgr: controller {ctrl} fw={ver} loaded {hours}h ago\n"
            "{ts4} {node} kernel: {dev}: same device on {node2} with fw=3.1.9 reports zero errors",

            # --- MEDIUM 2: looks like perf degradation, but version correlation ---
            "{ts1} {node} storage-mgr: {vol} latency p99={ms}ms (baseline: {baseline_ms}ms)\n"
            "{ts2} {node} ctrl-mgr: controller {ctrl} queue depth={count}, service time elevated\n"
            "{ts3} {node} ctrl-mgr: controller {ctrl} fw={ver} loaded {hours}h ago\n"
            "{ts4} {node} storage-mgr: latency elevation only on nodes running fw={ver}\n"
            "{ts5} {node} ctrl-mgr: vendor bulletin {incident}: scheduling regression in fw={ver}",

            # --- MEDIUM 3: looks like capacity issue, but version-dependent GC ---
            "{ts1} {node} storage-mgr: {pool} utilization={pct}% of {size_tb}TB\n"
            "{ts2} {node} storage-mgr: {pool} garbage collection backlog={count} segments\n"
            "{ts3} {node} ctrl-mgr: controller {ctrl} fw={ver} loaded {hours}h ago\n"
            "{ts4} {node} storage-mgr: {node2} at same utilization with fw=3.1.9 shows zero GC backlog\n"
            "{ts5} {node} ctrl-mgr: vendor confirmed: fw={ver} GC scheduling delay under high utilization",

            # --- HARD 1: looks like network issue -- retransmissions, but only on fw version ---
            "{ts1} {node} network-mgr: {port} retransmissions={rate}/s, baseline=0.01/s\n"
            "{ts2} {node} network-mgr: {port} CRC errors=0 in last {hours}h\n"
            "{ts3} {node} ctrl-mgr: controller {ctrl} fw={ver} loaded {hours}h ago\n"
            "{ts4} {node} network-mgr: {port} hardware diagnostics: OK\n"
            "{ts5} {node} network-mgr: retransmissions only on nodes running fw={ver}, other nodes on same fabric report normal",

            # --- HARD 2: looks like drive failure -- device resets, but version-dependent ---
            "{ts1} {node} kernel: {dev}: device not responding, resetting link\n"
            "{ts2} {node} kernel: {dev}: cmd 0x28 timeout after 30s\n"
            "{ts3} {node} smartd: Device /dev/{dev}: SMART Health Status: OK\n"
            "{ts4} {node} ctrl-mgr: controller {ctrl} fw={ver} loaded {hours}h ago\n"
            "{ts5} {node} kernel: same /dev/{dev} on {node2} with fw=3.1.9 operates without timeouts",
        ],
    },
}


def render_template(template_str):
    """Fill a template string with random values from SHARED_VARS.

    Uses a consistent set of variable picks per render so that the same
    {dev}, {node}, etc. are reused within one log entry.
    """
    # Pre-pick one value for each variable that might appear
    picks = {}
    for key, pool in SHARED_VARS.items():
        picks[key] = str(random.choice(pool))

    # Generate timestamps (ts1..ts5)
    base = datetime(2026, 3, 28, random.randint(0, 23), random.randint(0, 59), random.randint(0, 59))
    stamps = generate_timestamps(5, base=base, interval_sec=(1, 30))
    for i, ts in enumerate(stamps, start=1):
        picks[f"ts{i}"] = ts

    # Pick a second distinct node for {node2}
    node_pool = SHARED_VARS["node"] + SHARED_VARS["node2"]
    node2_val = str(random.choice([n for n in node_pool if n != picks["node"]]))
    picks["node2"] = node2_val

    # Format the template
    try:
        return template_str.format(**picks)
    except KeyError as e:
        raise KeyError(f"Template references undefined variable: {e}") from e


CATEGORIES = list(TEMPLATES.keys())
print(f"Categories ({len(CATEGORIES)}): {CATEGORIES}")
for cat, data in TEMPLATES.items():
    print(f"  {cat}: {len(data['templates'])} templates")
print(f"Total templates: {sum(len(d['templates']) for d in TEMPLATES.values())}")

## Step 3: Generate Synthetic Incident Report Dataset

Generate 720 incident reports (120 per category, 8 templates each = 15 per template) by filling multi-line log templates with randomized variable values. Each report is a realistic multi-line syslog entry that challenges classifiers.

In [ ]:
# Step 3: Generate Synthetic Incident Report Dataset
import math

# Generate 720 samples (120 per category, ceil(120/8) = 15 per template)
samples = []
samples_per_template = math.ceil(120 / 8)  # = 15

for cat in CATEGORIES:
    templates = TEMPLATES[cat]["templates"]
    for t_idx, template_str in enumerate(templates):
        for _ in range(samples_per_template):
            msg = render_template(template_str)
            samples.append({"text": msg, "label": cat})

random.shuffle(samples)
print(f"Generated {len(samples)} incident reports")
print(f"Categories: {dict(Counter(s['label'] for s in samples))}")

## Step 4: Explore the Data

Display 2 examples per category to see the multi-line log entries. Each sample contains multiple syslog lines separated by newlines. Notice how the same error type can be expressed with completely different vocabulary and how different categories share terminology like "latency," "timeout," "controller," and "firmware."

In [ ]:
# Step 4: Explore the Data
for cat in CATEGORIES:
    cat_samples = [s for s in samples if s['label'] == cat][:2]
    print(f"\n{'='*70}")
    print(f"  {cat}")
    print(f"{'='*70}")
    for i, s in enumerate(cat_samples):
        print(f"\n  Example {i+1}:")
        # Each sample is multi-line (\n-separated log lines)
        for log_line in s['text'].split('\n'):
            print(f"    {log_line}")
print()

## Step 5: Baseline — TF-IDF + XGBoost

First, we establish a traditional ML baseline. TF-IDF converts text to word-frequency features, then XGBoost classifies. This is a strong baseline for text classification — but it treats each message as a bag of words, ignoring word order and context.

In [ ]:
# Step 5: Baseline — TF-IDF + XGBoost
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import classification_report, accuracy_score
from xgboost import XGBClassifier
import time

# Prepare data
texts = [s['text'] for s in samples]
labels = [s['label'] for s in samples]

le = LabelEncoder()
y = le.fit_transform(labels)

# Train/test split
texts_train, texts_test, y_train, y_test = train_test_split(
    texts, y, test_size=0.2, random_state=42, stratify=y
)

# TF-IDF vectorization
print("Vectorizing with TF-IDF...")
tfidf = TfidfVectorizer(max_features=5000, ngram_range=(1, 2), stop_words='english')
X_train_tfidf = tfidf.fit_transform(texts_train)
X_test_tfidf = tfidf.transform(texts_test)

print(f"Vocabulary size: {len(tfidf.vocabulary_)}")
print(f"Feature matrix: {X_train_tfidf.shape}")

# Train XGBoost
print("\nTraining XGBoost on TF-IDF features...")
t0 = time.time()
xgb_text = XGBClassifier(n_estimators=200, max_depth=6, random_state=42,
                          eval_metric='mlogloss')
xgb_text.fit(X_train_tfidf, y_train)
xgb_train_time = time.time() - t0

# Evaluate
xgb_preds = xgb_text.predict(X_test_tfidf)
xgb_accuracy = accuracy_score(y_test, xgb_preds)

print(f"\nTraining time: {xgb_train_time:.2f}s")
print(f"Accuracy: {xgb_accuracy:.1%}")
print(f"\nClassification Report:")
print(classification_report(y_test, xgb_preds, target_names=le.classes_))

## Step 6: Why XGBoost Struggles Here

The accuracy above is decent but limited. Three common failure modes:

1. **Vocabulary overlap:** "latency," "timeout," "controller," and "firmware" appear in 3+ categories — TF-IDF cannot distinguish context
2. **Negation context:** "No device errors — timeouts caused by space exhaustion" contains drive-failure vocabulary but the meaning is capacity warning
3. **Cross-category causation:** "Cache destage failed to device" sounds like firmware/performance but is actually a drive failure causing the cache issue

An LLM processes the full message as a sequence, understanding context, negation, and causal relationships rather than just word frequency.

## Step 7: Prepare SFT Training Data

Format each sample as a prompt/completion pair for supervised fine-tuning. The prompt includes the full incident report and asks for a classification; the completion is the category label. This is the same format used in the Post-Training Pipeline.

In [ ]:
# Step 7: Prepare SFT Training Data
from datasets import Dataset
from sklearn.model_selection import train_test_split as tts2

def format_for_sft(sample):
    prompt = (
        f"Classify the following storage error log message into one of these categories: "
        f"{', '.join(CATEGORIES)}.\n\n"
        f"Log message: {sample['text']}\n\n"
        f"Classification:"
    )
    completion = f" {sample['label']}"
    return {"prompt": prompt, "completion": completion}

all_formatted = [format_for_sft(s) for s in samples]

# Split formatted data the same way as Step 5
random.seed(42)
np.random.seed(42)
fmt_train, fmt_test = tts2(all_formatted, test_size=0.2, random_state=42,
                            stratify=[s['label'] for s in samples])

train_dataset = Dataset.from_dict({
    "prompt": [s["prompt"] for s in fmt_train],
    "completion": [s["completion"] for s in fmt_train],
})

print(f"SFT training samples: {len(train_dataset)}")
print(f"\nExample training input:")
print(f"  Prompt (first 300 chars): {fmt_train[0]['prompt'][:300]}...")
print(f"  Completion: {fmt_train[0]['completion']}")

## Step 8A: Full Training Mode (skip for demo)

Train a LoRA adapter on SmolLM2-360M from scratch. This takes ~20 minutes on a T4 GPU. After training, the adapter is saved locally and automatically uploaded to the HuggingFace Hub if `HF_TOKEN` was set in Step 1.

**Skip this cell** if you want a quick demo — use Step 8B instead to load a pre-trained adapter.

In [ ]:
# Step 8A: Full Training Mode (~20 min on T4)
# SKIP THIS CELL for demo mode — use Step 8B instead

if not torch.cuda.is_available():
    raise RuntimeError("GPU required for training. Go to Runtime > Change runtime type > T4 GPU")

from transformers import AutoModelForCausalLM, AutoTokenizer, set_seed
from peft import LoraConfig, get_peft_model
from trl import SFTTrainer, SFTConfig

set_seed(42)

MODEL_NAME = "HuggingFaceTB/SmolLM2-360M"
device = "cuda" if torch.cuda.is_available() else "cpu"

print(f"Loading {MODEL_NAME}...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME, dtype=torch.bfloat16, device_map="auto"
)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

# LoRA configuration — same settings as Post-Training Pipeline
lora_config = LoraConfig(
    r=32,
    lora_alpha=64,
    target_modules=["q_proj", "v_proj", "k_proj", "o_proj"],
    lora_dropout=0.05,
    task_type="CAUSAL_LM",
)

model = get_peft_model(model, lora_config)
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total = sum(p.numel() for p in model.parameters())
print(f"Trainable parameters: {trainable:,} / {total:,} ({100*trainable/total:.2f}%)")

# Training configuration
training_args = SFTConfig(
    output_dir="/tmp/sft_error_logs",
    num_train_epochs=6,
    per_device_train_batch_size=4,
    gradient_accumulation_steps=2,
    learning_rate=1e-4,
    lr_scheduler_type="cosine",
    warmup_ratio=0.1,
    logging_steps=10,
    save_strategy="no",
    bf16=torch.cuda.is_available(),
    report_to="none",
)

trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    processing_class=tokenizer,
)

print("\nStarting SFT training...")
t0 = time.time()
trainer.train()
sft_train_time = time.time() - t0
print(f"\nTraining completed in {sft_train_time/60:.1f} minutes")

# Save adapter locally
model.save_pretrained("/tmp/sft_error_logs/final_adapter")
tokenizer.save_pretrained("/tmp/sft_error_logs/final_adapter")
print("Adapter saved to /tmp/sft_error_logs/final_adapter")

# Upload to HuggingFace Hub
if HF_TOKEN:
    import os
    cache_path = os.path.expanduser("~/.cache/huggingface/token")
    if os.path.exists(cache_path):
        os.remove(cache_path)
    from huggingface_hub import login
    login(token=HF_TOKEN, add_to_git_credential=False)
    model.push_to_hub(HF_REPO, token=HF_TOKEN)
    tokenizer.push_to_hub(HF_REPO, token=HF_TOKEN)
    print(f"Uploaded to {HF_REPO}")
else:
    print("No HF_TOKEN set in Step 1 — skipping upload.")

## Step 8B: Demo Mode — Load Pre-trained Adapter

Load a pre-trained LoRA adapter from the HuggingFace Hub. This skips the ~20 minute training step and lets you jump straight to evaluation. The adapter was trained with the exact same configuration as Step 8A.

In [ ]:
# Step 8B: Demo Mode — Load Pre-trained Adapter from HuggingFace Hub
# USE THIS CELL instead of Step 8A for a quick demo

from transformers import AutoModelForCausalLM, AutoTokenizer, set_seed
from peft import PeftModel

set_seed(42)

if not torch.cuda.is_available():
    raise RuntimeError("GPU required for inference. Go to Runtime > Change runtime type > T4 GPU")

MODEL_NAME = "HuggingFaceTB/SmolLM2-360M"
device = "cuda" if torch.cuda.is_available() else "cpu"

print(f"Loading base model: {MODEL_NAME}...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME, dtype=torch.bfloat16, device_map="auto"
)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

print(f"Loading pre-trained adapter: {HF_REPO}...")
model = PeftModel.from_pretrained(base_model, HF_REPO)
print("Adapter loaded successfully.")

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total = sum(p.numel() for p in model.parameters())
print(f"Parameters: {trainable:,} trainable / {total:,} total")

# Mark that we skipped training
sft_train_time = 0

## Step 9: Evaluate the SFT Model

Run inference on the test set and compare predictions against ground truth. Each sample is formatted with the same prompt template used during training.

In [ ]:
# Step 9: Evaluate the SFT Model
model.eval()

# Reconstruct test set
_, test_samples_eval = tts2(samples, test_size=0.2, random_state=42,
                             stratify=[s['label'] for s in samples])

sft_preds = []
sft_correct = 0
total_test = len(test_samples_eval)

print(f"Evaluating SFT model on {total_test} test samples...")
for idx, sample in enumerate(test_samples_eval):
    if (idx + 1) % 20 == 0 or idx == 0:
        print(f"  Progress: {idx+1}/{total_test}")

    prompt = (
        f"Classify the following storage error log message into one of these categories: "
        f"{', '.join(CATEGORIES)}.\n\n"
        f"Log message: {sample['text']}\n\n"
        f"Classification:"
    )
    inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=512).to(device)

    with torch.no_grad():
        output_ids = model.generate(
            **inputs, max_new_tokens=20, do_sample=False,
            pad_token_id=tokenizer.pad_token_id
        )

    generated = tokenizer.decode(output_ids[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True).strip()

    # Check if any category name appears in the output
    predicted = None
    for cat in CATEGORIES:
        if cat.lower() in generated.lower():
            predicted = cat
            break

    is_correct = predicted == sample['label']
    if is_correct:
        sft_correct += 1
    sft_preds.append({
        'true': sample['label'],
        'predicted': predicted or generated[:50],
        'raw_output': generated[:100],
        'correct': is_correct,
    })

sft_accuracy = sft_correct / len(test_samples_eval)
print(f"\nSFT Accuracy: {sft_accuracy:.1%} ({sft_correct}/{len(test_samples_eval)})")

# Per-category breakdown
print(f"\nPer-Category Results:")
print("-" * 50)
for cat in CATEGORIES:
    cat_preds = [p for p in sft_preds if p['true'] == cat]
    cat_correct = sum(1 for p in cat_preds if p['correct'])
    cat_total = len(cat_preds)
    pct = cat_correct / cat_total if cat_total > 0 else 0
    bar = "#" * int(pct * 10) + "." * (10 - int(pct * 10))
    print(f"  {cat:<25s} {bar} {cat_correct}/{cat_total} ({pct:.0%})")

## Step 10: Head-to-Head Comparison

Side-by-side charts comparing TF-IDF + XGBoost vs SFT fine-tuning on overall accuracy and per-category accuracy, plus a summary table with training time, model size, and qualitative differences.

In [ ]:
# Step 10: Head-to-Head Comparison
import matplotlib.pyplot as plt
import pandas as pd

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Left: Accuracy comparison
models = ['TF-IDF + XGBoost', 'SFT (SmolLM2-360M)']
accs = [xgb_accuracy, sft_accuracy]
colors = ['#f97316', '#3b82f6']
bars = ax1.bar(models, accs, color=colors, alpha=0.8, width=0.5)
ax1.set_ylabel('Accuracy')
ax1.set_title('Text Classification Accuracy')
ax1.set_ylim(0, 1.1)
for bar, val in zip(bars, accs):
    ax1.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.02,
             f'{val:.1%}', ha='center', fontweight='bold', fontsize=13)

# Right: Per-category comparison
xgb_cat_acc = {}
sft_cat_acc = {}
for cat in CATEGORIES:
    # XGBoost
    cat_mask = [le.classes_[y_test[i]] == cat for i in range(len(y_test))]
    cat_indices = [i for i, m in enumerate(cat_mask) if m]
    if cat_indices:
        xgb_cat_acc[cat] = sum(1 for i in cat_indices if xgb_preds[i] == y_test[i]) / len(cat_indices)
    # SFT
    cat_sft = [p for p in sft_preds if p['true'] == cat]
    if cat_sft:
        sft_cat_acc[cat] = sum(1 for p in cat_sft if p['correct']) / len(cat_sft)

x = np.arange(len(CATEGORIES))
width = 0.35
ax2.bar(x - width/2, [xgb_cat_acc.get(c, 0) for c in CATEGORIES], width,
        label='TF-IDF + XGBoost', color='#f97316', alpha=0.8)
ax2.bar(x + width/2, [sft_cat_acc.get(c, 0) for c in CATEGORIES], width,
        label='SFT', color='#3b82f6', alpha=0.8)
ax2.set_ylabel('Accuracy')
ax2.set_title('Per-Category Accuracy')
ax2.set_xticks(x)
ax2.set_xticklabels(CATEGORIES, rotation=35, ha='right', fontsize=8)
ax2.legend()
ax2.set_ylim(0, 1.15)

plt.tight_layout()
plt.show()

# Summary table
print("\n" + "=" * 70)
print("  COMPARISON: TF-IDF + XGBoost vs SFT Fine-Tuning")
print("=" * 70)
if sft_train_time > 0:
    sft_time_str = f'{sft_train_time/60:.1f} min'
else:
    sft_time_str = 'Pre-trained (skipped)'

comparison_data = {
    'Metric': ['Accuracy', 'Training Time', 'Model Size', 'Hardware', 'Handles Novel Phrasing', 'Contextual Understanding'],
    'TF-IDF + XGBoost': [f'{xgb_accuracy:.1%}', f'{xgb_train_time:.1f}s', '~5 MB', 'CPU', 'Limited', 'None (bag of words)'],
    'SFT (SmolLM2-360M)': [f'{sft_accuracy:.1%}', sft_time_str, '~700 MB', 'GPU', 'Good', 'Full sequence modeling'],
}
comparison_df = pd.DataFrame(comparison_data)
print(comparison_df.to_string(index=False))
print("=" * 70)

## Step 11: Error Analysis

Examine cases where the SFT model succeeds and TF-IDF + XGBoost fails. This highlights the LLM's advantage in understanding contextual meaning.

In [ ]:
# Step 11: Error Analysis — Where does the LLM win?
print("Cases where SFT succeeds and TF-IDF + XGBoost fails:")
print("=" * 70)

# Align test samples between the two models
sft_test = test_samples_eval
count = 0
for i, (sample, sft_pred) in enumerate(zip(sft_test, sft_preds)):
    if i >= len(y_test):
        break
    xgb_pred_label = le.classes_[xgb_preds[i]] if i < len(xgb_preds) else None
    xgb_wrong = xgb_pred_label != sample['label']
    sft_right = sft_pred['correct']
    
    if xgb_wrong and sft_right and count < 4:
        count += 1
        print(f"\n  Example {count}:")
        print(f"  True label:    {sample['label']}")
        print(f"  XGBoost pred:  {xgb_pred_label} X")
        print(f"  SFT pred:      {sft_pred['predicted']} OK")
        text = sample['text'][:150]
        print(f"  Log message:   {text}...")

if count == 0:
    print("  (No clear examples in this run -- both models may have similar error patterns)")

print(f"\n\nSummary: SFT's advantage comes from understanding the *meaning* of incident reports,")
print(f"not just keyword frequency. This matters most when:")
print(f"  - Different categories share vocabulary (latency, timeout, controller, firmware)")
print(f"  - Negation changes meaning (no device errors, not a firmware defect)")
print(f"  - Cross-category causation requires reasoning (cache failed because drive failed)")

## Step 12: Conclusion — A Framework for Choosing Your Approach

| Characteristic | Use Traditional ML | Use LLM Fine-Tuning |
|---|---|---|
| **Input format** | Structured numbers, tables | Unstructured text, natural language |
| **Feature engineering** | Clear, known features | Features hard to extract manually |
| **Vocabulary** | N/A or controlled | Variable, natural language |
| **Context dependency** | Low — features are independent | High — meaning depends on context |
| **Training resources** | CPU, seconds-minutes | GPU, minutes-hours |
| **Model size** | KBs-MBs | Hundreds of MBs |
| **Interpretability** | High (feature importance) | Low (black box) |

**The two notebooks together give you a decision framework:**
- **Structured numeric data** (I/O metrics, sensor readings, performance counters) -> Random Forest / XGBoost
- **Unstructured text** (logs, error messages, incident reports, documentation) -> LLM fine-tuning

**Neither approach is universally better.** The right choice depends on your data, not on what's trending.

Return to the [Post-Training Pipeline](Post_Training_Pipeline.ipynb) to run the full SFT -> DPO -> GRPO pipeline, or see the [Traditional ML Comparison](Traditional_ML_Comparison.ipynb) for the structured data perspective.